# Day 18/42: Overfitting vs Underfitting

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VaishnaviJagtap18/42-days-aiml-challenge/blob/main/week3_core_ml/day18_overfitting_underfitting/day18_notebook.ipynb)

## What You'll Learn

- Why a high training score does not mean your model is good
- How to spot overfitting and underfitting from train vs validation scores
- The bias-variance tradeoff, shown with real numbers and real plots
- How to read a learning curve
- How regularization fixes overfitting, and how much is too much

## Datasets used

- A small synthetic dataset (regression) built in this notebook, no download needed
- `load_breast_cancer`, a built-in scikit-learn dataset, no internet required

Run every cell top to bottom. Nothing here needs an internet connection or an API key.

## Setup

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import r2_score, accuracy_score

np.random.seed(42)
print("Setup complete. Libraries loaded.")

## The Concept

A model that scores 99% on training data and 60% on new data has not learned the pattern. It has memorized the training set.

**Underfitting**: the model is too simple to capture the real pattern. Low accuracy on training data AND low accuracy on validation data.

**Overfitting**: the model is complex enough to fit noise along with the real pattern. High accuracy on training data, low accuracy on validation data.

**Good fit**: training and validation scores are both reasonably high, and close to each other.

This is the bias-variance tradeoff. High bias shows up as underfitting. High variance shows up as overfitting. The model you want sits between the two, where total error from both sources is lowest, not where training error is lowest.

## 1. Seeing It Happen: Three Models, One Dataset

We'll fit the same data with three models of increasing complexity: a straight line, a degree-4 polynomial, and a degree-15 polynomial. Same data every time. Only the model complexity changes.

In [ ]:
def true_function(x):
    return np.sin(1.5 * np.pi * x)

n_samples = 60
X = np.sort(np.random.rand(n_samples))
y = true_function(X) + np.random.randn(n_samples) * 0.2

X_train, X_test, y_train, y_test = train_test_split(
    X.reshape(-1, 1), y, test_size=0.3, random_state=42
)

print("Train size:", X_train.shape[0])
print("Test size:", X_test.shape[0])

In [ ]:
degrees = [1, 4, 15]
labels = ["Underfit (degree 1)", "Good fit (degree 4)", "Overfit (degree 15)"]
colors = ["#E53935", "#00C853", "#7C4DFF"]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
X_plot = np.linspace(0, 1, 200).reshape(-1, 1)
results = []

for ax, degree, label, color in zip(axes, degrees, labels, colors):
    model = make_pipeline(PolynomialFeatures(degree), LinearRegression())
    model.fit(X_train, y_train)

    train_r2 = r2_score(y_train, model.predict(X_train))
    test_r2 = r2_score(y_test, model.predict(X_test))

    results.append({
        "model": label,
        "degree": degree,
        "train_r2": round(train_r2, 3),
        "test_r2": round(test_r2, 3),
        "gap (train - test)": round(train_r2 - test_r2, 3)
    })

    ax.scatter(X_train, y_train, color="gray", s=20, label="train")
    ax.scatter(X_test, y_test, color="black", s=20, marker="x", label="test")
    ax.plot(X_plot, model.predict(X_plot), color=color, linewidth=2)
    ax.set_title(label, fontsize=10)
    ax.set_ylim(-2, 2)
    ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

results_df = pd.DataFrame(results)
results_df

**Read the chart left to right.**

The degree-1 line cannot bend, so it misses the curve in the data. That is underfitting: both train and test R2 are low.

The degree-4 curve follows the real shape of the data without chasing every noisy point. Train and test R2 are both high and close together. That is the fit you want.

The degree-15 curve bends itself into knots trying to pass through every single training point, including the noise. Train R2 looks great. Test R2 falls off a cliff. That is overfitting, and it's the model that would look best on a leaderboard and worst in production.

## 2. The Same Problem on a Real Dataset

Synthetic data makes the idea easy to see. Here it is again on a real medical dataset (`load_breast_cancer`, built into scikit-learn), using a decision tree where `max_depth` controls how complex the model is allowed to get.

In [ ]:
data = load_breast_cancer()
Xc, yc = data.data, data.target

Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    Xc, yc, test_size=0.3, random_state=42, stratify=yc
)

depths = range(1, 21)
train_acc_list, test_acc_list = [], []

for depth in depths:
    clf = DecisionTreeClassifier(max_depth=depth, random_state=42)
    clf.fit(Xc_train, yc_train)
    train_acc_list.append(accuracy_score(yc_train, clf.predict(Xc_train)))
    test_acc_list.append(accuracy_score(yc_test, clf.predict(Xc_test)))

depth_df = pd.DataFrame({
    "max_depth": list(depths),
    "train_accuracy": train_acc_list,
    "test_accuracy": test_acc_list
})
depth_df["gap"] = depth_df["train_accuracy"] - depth_df["test_accuracy"]
depth_df.round(3)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(depth_df["max_depth"], depth_df["train_accuracy"], marker="o", label="Train accuracy", color="#7C4DFF")
plt.plot(depth_df["max_depth"], depth_df["test_accuracy"], marker="o", label="Test accuracy", color="#00C4CC")
plt.xlabel("max_depth")
plt.ylabel("Accuracy")
plt.title("Train vs Test Accuracy as Tree Depth Increases (Breast Cancer Dataset)")
plt.legend()
plt.tight_layout()
plt.show()

best_depth = depth_df.loc[depth_df["test_accuracy"].idxmax(), "max_depth"]
print(f"Best test accuracy happens at max_depth = {best_depth}")
print("Past that point, train accuracy keeps climbing to 1.0 while test accuracy flattens or drops.")

## 3. Learning Curves: The Tool in Today's ML Spotlight

`learning_curve` from scikit-learn trains a model on increasing slices of the training data and records train and validation scores at each size. Run it on an overfit model (`max_depth=15`) and the train-validation gap is visible immediately, no guessing required.

In [ ]:
overfit_model = DecisionTreeClassifier(max_depth=15, random_state=42)

train_sizes, train_scores, val_scores = learning_curve(
    overfit_model, Xc, yc, cv=5,
    train_sizes=np.linspace(0.1, 1.0, 8),
    scoring="accuracy", random_state=42
)

train_mean = train_scores.mean(axis=1)
val_mean = val_scores.mean(axis=1)

plt.figure(figsize=(8, 5))
plt.plot(train_sizes, train_mean, marker="o", label="Train score", color="#7C4DFF")
plt.plot(train_sizes, val_mean, marker="o", label="Validation score", color="#00C4CC")
plt.fill_between(train_sizes, train_mean, val_mean, color="#E53935", alpha=0.08)
plt.xlabel("Training set size")
plt.ylabel("Accuracy")
plt.title("Learning Curve: Decision Tree (max_depth=15)")
plt.legend()
plt.tight_layout()
plt.show()

print("Train score at every size:", np.round(train_mean, 3))
print("Validation score at every size:", np.round(val_mean, 3))

The train line sits at a perfect 1.0 the entire time, no matter how much data you give it. That's the signature of a model with more capacity than it needs, memorizing whatever it sees. The validation line improves as training data grows, which tells you more data would help, but the gap between the two lines tells you the model itself is also part of the problem.

## 4. Fixing Overfitting: Regularization in Action

Back to the degree-15 polynomial from Section 1. Instead of plain `LinearRegression`, we'll use `Ridge`, which penalizes large coefficients. The `alpha` parameter controls how hard that penalty is.

In [ ]:
alphas = [0, 0.0001, 0.001, 0.01, 0.1]
reg_results = []

for alpha in alphas:
    if alpha == 0:
        model = make_pipeline(PolynomialFeatures(15), LinearRegression())
    else:
        model = make_pipeline(PolynomialFeatures(15), Ridge(alpha=alpha))
    model.fit(X_train, y_train)
    train_r2 = r2_score(y_train, model.predict(X_train))
    test_r2 = r2_score(y_test, model.predict(X_test))
    reg_results.append({
        "alpha": alpha,
        "train_r2": round(train_r2, 3),
        "test_r2": round(test_r2, 3),
        "gap": round(train_r2 - test_r2, 3)
    })

pd.DataFrame(reg_results)

At `alpha = 0` (no regularization), the gap is large: this is the overfit model from Section 1. Add a small penalty and the gap nearly disappears, with test R2 jumping close to train R2. Push `alpha` too high and both scores drop together: that's regularization tipping the model back into underfitting. The right amount of regularization is a search, not a guess, which is exactly what `GridSearchCV` is for (Day 20).

## 5. Practice: Diagnose the Model

Five models below report only their train and validation accuracy. Before running the next cell, write down your own diagnosis for each: underfitting, overfitting, or good fit.

In [ ]:
scenarios = [
    {"name": "Model A", "train_acc": 0.99, "val_acc": 0.97},
    {"name": "Model B", "train_acc": 0.65, "val_acc": 0.64},
    {"name": "Model C", "train_acc": 0.98, "val_acc": 0.71},
    {"name": "Model D", "train_acc": 0.80, "val_acc": 0.79},
    {"name": "Model E", "train_acc": 0.92, "val_acc": 0.60},
]

for s in scenarios:
    print(f"{s['name']}: train = {s['train_acc']}, val = {s['val_acc']}")

**Your turn.** Write your diagnosis for each model here before checking the solution below.

- Model A: 
- Model B: 
- Model C: 
- Model D: 
- Model E: 

### Solution

In [ ]:
def diagnose(train_acc, val_acc, gap_threshold=0.08, low_threshold=0.75):
    gap = train_acc - val_acc
    if train_acc < low_threshold and val_acc < low_threshold:
        return "Underfitting (both scores low)"
    elif gap > gap_threshold:
        return "Overfitting (big train-val gap)"
    else:
        return "Good fit (small gap, decent scores)"

print(f"{'Model':<10}{'Train':<8}{'Val':<8}{'Diagnosis'}")
for s in scenarios:
    diagnosis = diagnose(s["train_acc"], s["val_acc"])
    print(f"{s['name']:<10}{s['train_acc']:<8}{s['val_acc']:<8}{diagnosis}")

Notice Model A: a 99% training score, the kind of number that looks like a red flag on its own. But validation is 97%, a 2-point gap. That's a good fit, not overfitting. The gap between train and validation tells you more than either number alone. This is the mistake beginners make most often: treating a high train score as automatically suspicious, when the real signal is the gap.

## 6. Try It Yourself

No solution provided here. Make these changes and watch what happens to the train/test gap:

1. In Section 2, change `max_depth` to just `[1, 3, 5, 10, 20]` and re-plot. Does the test accuracy peak move?
2. In Section 1, change `n_samples` from 60 to 300 and re-run the degree-15 model. Does more data shrink the overfitting gap, even with the same model complexity?
3. Add a sixth scenario to the practice exercise with `train_acc=0.70` and `val_acc=0.72`. What does the function return, and does that diagnosis make sense to you?

## Self-Check Before Day 19

You're ready to move on if you can answer these without scrolling back up:

1. A model scores 95% on training data and 94% on validation data. Is this overfitting?
2. Name two ways to fix an overfitting model.
3. Name one way to fix an underfitting model.
4. What does a learning curve show that a single accuracy number cannot?
5. Why is `alpha=0.1` worse than `alpha=0.001` in the Ridge example above, even though both are "regularized"?

If any of these feel shaky, re-run the relevant section above before starting Day 19.

## What's Next

Tomorrow, Day 19: Model Evaluation. Accuracy alone hides exactly the kind of problem you saw with Model A and Model E in this notebook. We'll go through Precision, Recall, F1, and ROC-AUC, and when accuracy actively lies to you.

Full series repo: github.com/VaishnaviJagtap18/42-days-aiml-challenge

#42DaysOfML #MachineLearning #MLEngineer #Python #DataScience